# Mapa de risco de inundação com o método HAND (Height Above Nearest Drainage)

Este notebook converte um *script* em um fluxo reprodutível e didático em Jupyter Notebook.
Todo o texto explicativo está em **português** e o foco é **entender e reproduzir o processo**, não otimizar desempenho.

## Objetivo
A partir de um Modelo Digital de Elevação (MDE/DEM) e, opcionalmente, uma máscara/limite da área de estudo, vamos:
1. Preparar o DEM (projeção, recorte e correções hidrológicas).
2. Calcular direção de fluxo e acumulação.
3. Extrair a rede de drenagem (canais) com um limiar de acumulação.
4. Calcular o **HAND**: altura de cada célula em relação ao canal de drenagem mais próximo (ao longo do escoamento).
5. (Opcional) Classificar o HAND em classes de suscetibilidade/risco e exportar os resultados.

> **Nota importante:** como eu não recebi o seu script original nesta mensagem, este notebook foi estruturado para ser um *equivalente funcional* do fluxo HAND, com pontos claramente marcados para você colar/ajustar trechos específicos do seu código (paths, parâmetros, etc.).
Se você colar o script, eu consigo adaptar 1:1 (mesmas bibliotecas, variáveis e saídas).

## 1) Ambiente e dependências
O método HAND pode ser implementado com diferentes bibliotecas. Aqui usamos uma combinação comum:
- **rasterio**: leitura/escrita de rasters (GeoTIFF).
- **geopandas**: leitura de vetores (shapefile/GeoPackage) e recorte.
- **numpy**: operações matriciais.
- **pysheds** *(opcional, recomendado)*: ferramentas hidrológicas (direção de fluxo, acumulação, etc.).

Se você já tem um stack diferente (por exemplo, WhiteboxTools, RichDEM, GRASS, TauDEM), substitua as células correspondentes.

In [ ]:
pip install pysheds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.2 MB/s eta 0:00:00


In [ ]:
# Se necessário, instale dependências (descomente conforme o seu ambiente)
# !pip install rasterio geopandas shapely pyproj numpy matplotlib pysheds

import os
from pathlib import Path
import sys

import numpy as np
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import matplotlib.pyplot as plt
from pysheds.grid import Grid
import numpy as np
import rasterio

# pysheds é opcional; caso não esteja disponível, a célula abaixo avisará.
try:
    from pysheds.grid import Grid
    PYSheds_OK = True
except Exception as e:
    PYSheds_OK = False
    print("Aviso: pysheds não está disponível. Para um fluxo HAND completo, instale pysheds.")
    print("Erro:", e)


## 2) Entradas e parâmetros
Nesta etapa definimos os caminhos dos arquivos e parâmetros principais.

### Arquivos típicos
- `dem_path`: DEM em GeoTIFF (ou outro formato raster suportado).
- `aoi_path` (opcional): polígono da área de estudo (AOI) para recorte.

### Parâmetros-chave
- `acc_threshold`: limiar de acumulação de fluxo para definir canais (quanto maior, menos canais).
- `hand_classes_m`: classes em metros para categorizar risco/suscetibilidade (exemplo; ajuste ao seu caso).

In [ ]:
# ====== AJUSTE AQUI ======
data = '/home/jorge/Downloads/IBGE/data'
outputs = '/home/jorge/Downloads/IBGE/result/'

DATA_DIR = Path(data)               # pasta de dados
OUT_DIR  = Path(outputs)            # pasta de saída
OUT_DIR.mkdir(parents=True, exist_ok=True)

# dem_path = DATA_DIR / "MDE_ROI.tif"                         # DEM (GeoTIFF)
dem_path = DATA_DIR / 'MED_ROI_Clipped.tif'
aoi_path = DATA_DIR / "03_reproyect_AOI_area_drenagem"      # opcional: polígono da área de estudo (GPKG/SHP)
aoi_layer = None                      # se GPKG tiver múltiplas camadas, defina o nome

acc_threshold = 1000                  # limiar de acumulação (ajuste conforme resolução do DEM e escala)
hand_classes_m = [0, 2, 5, 10, 20, 50, 1e9]  # classes (m): 0–2, 2–5, 5–10, 10–20, 20–50, >50


## 3) Leitura do DEM e (opcional) recorte pela área de estudo
Se você tiver um limite municipal/bacia/recorte, aplicamos a máscara vetorial ao raster.
Isso reduz custo computacional e garante que as estatísticas se refiram apenas à área de interesse.

Se não houver AOI, esta etapa apenas lê o DEM inteiro.

In [ ]:
def read_dem(dem_fp: Path):
    with rasterio.open(dem_fp) as src:
        dem = src.read(1, masked=True).astype("float32")
        profile = src.profile.copy()
    return dem, profile

def clip_dem_to_aoi(dem_fp: Path, aoi_fp: Path, layer=None):
    aoi = gpd.read_file(aoi_fp, layer=layer)
    aoi = aoi.to_crs(rasterio.open(dem_fp).crs)  # garantir CRS consistente

    geoms = [geom for geom in aoi.geometry if geom is not None]
    with rasterio.open(dem_fp) as src:
        out_img, out_transform = mask(src, geoms, crop=True)
        out_dem = out_img[0].astype("float32")
        out_profile = src.profile.copy()
        out_profile.update({
            "height": out_dem.shape[0],
            "width": out_dem.shape[1],
            "transform": out_transform
        })
    # Converte nodata em masked array
    nodata = out_profile.get("nodata", None)
    if nodata is not None:
        out_dem = np.ma.masked_equal(out_dem, nodata)
    else:
        out_dem = np.ma.masked_invalid(out_dem)
    return out_dem, out_profile, aoi

USE_AOI = aoi_path.exists()

if USE_AOI:
    dem, dem_profile, aoi = clip_dem_to_aoi(dem_path, aoi_path, layer=aoi_layer)
    print("DEM recortado:", dem.shape)
else:
    dem, dem_profile = read_dem(dem_path)
    print("DEM inteiro:", dem.shape)

print("CRS:", dem_profile.get("crs"))
print("Resolução:", dem_profile.get("transform").a, dem_profile.get("transform").e)


RasterioIOError: /home/jorge/Downloads/IBGE/data/MED_ROI_Clipped.tif: No such file or directory

### Visualização rápida do DEM
A inspeção visual ajuda a identificar problemas óbvios: buracos, ruído, offsets e bordas do recorte.

In [ ]:
# plt.figure(figsize=(8, 6))
# plt.imshow(dem, cmap="terrain")
# plt.colorbar(label="Elevação (m)")
# plt.title("DEM (recortado, se AOI foi fornecida)")
# plt.tight_layout()
# plt.show()


## 4) Correções hidrológicas do DEM
Modelos hidrológicos em grade exigem um DEM sem inconsistências que interrompam o escoamento.
O passo clássico é o **preenchimento de depressões (sink filling)** e, em alguns casos, **breaching**.

Aqui usamos as rotinas do `pysheds` quando disponível. Se você usa outra ferramenta (WhiteboxTools/GRASS), substitua esta seção.

In [ ]:

if not PYSheds_OK:
    raise RuntimeError(
        "Para seguir com este notebook, instale pysheds ou substitua esta seção pelo seu método de correção hidrológica."
    )

def _grid_keys(g):
    return list(getattr(g, "grids", {}).keys()) + list(getattr(g, "_grids", {}).keys())

def _get_any_layer(g, preferred_name=None, before_keys=None):
    """Recupera uma camada Raster instance do Grid de forma robusta (sGrid/versões antigas)."""
    # 1) nome preferido (atributo)
    if preferred_name and hasattr(g, preferred_name):
        return getattr(g, preferred_name)
    # 2) nome preferido (dicts)
    if preferred_name and hasattr(g, "grids") and preferred_name in g.grids:
        return g.grids[preferred_name]
    if preferred_name and hasattr(g, "_grids") and preferred_name in g._grids:
        return g._grids[preferred_name]

    # 3) detectar nova camada criada (diferença de chaves)
    if before_keys is not None:
        after = set(_grid_keys(g))
        new = list(after - set(before_keys))
        if len(new) > 0:
            k = new[0]
            if hasattr(g, "grids") and k in getattr(g, "grids", {}):
                return g.grids[k]
            if hasattr(g, "_grids") and k in getattr(g, "_grids", {}):
                return g._grids[k]

    return None

def _add_layer_from_array(g, name, arr, template_raster):
    """Se a função retorna ndarray, registra como camada no Grid (se suportado)."""
    arr = np.asarray(arr)
    if hasattr(g, "add_gridded_data"):
        g.add_gridded_data(
            arr,
            data_name=name,
            affine=getattr(template_raster, "affine", getattr(g, "affine", None)),
            crs=getattr(template_raster, "crs", getattr(g, "crs", None)),
            nodata=getattr(template_raster, "nodata", None),
        )
        return _get_any_layer(g, preferred_name=name)
    if hasattr(g, "add_data"):
        try:
            g.add_data(name, arr)
        except TypeError:
            g.add_data(arr, name=name)
        return _get_any_layer(g, preferred_name=name)
    raise RuntimeError("Esta versão do pysheds não permite registrar arrays como camada (sem add_gridded_data/add_data).")

# ------------------------------------------------------------
# 1) Escolher o raster real para o Grid (original ou recortado)
# ------------------------------------------------------------
if USE_AOI:
    tmp_clip = OUT_DIR / "_dem_clip_tmp.tif"
    prof = dem_profile.copy()
    prof.update(dtype="float32", count=1)
    nodata_val = prof.get("nodata", -9999)

    with rasterio.open(tmp_clip, "w", **prof) as dst:
        dst.write(np.asarray(dem.filled(nodata_val), dtype="float32"), 1)

    dem_for_grid = tmp_clip
else:
    dem_for_grid = dem_path

# ------------------------------------------------------------
# 2) Criar Grid + carregar DEM (retorna Raster instance)
# ------------------------------------------------------------
grid = Grid()
dem_raw_r = grid.read_raster(str(dem_for_grid), data_name="dem_raw")

# view() recebe Raster instance (não string)
dem_raw = grid.view(dem_raw_r)

# ------------------------------------------------------------
# 3) Correções hidrológicas (compatível com sGrid antigo)
#    - Não confiar em out_name: pode NÃO registrar no grid.
#    - Capturar retorno e/ou detectar camadas novas.
# ------------------------------------------------------------
before = _grid_keys(grid)
try:
    res_fill = grid.fill_depressions(dem=dem_raw_r, out_name="dem_filled")
except TypeError:
    res_fill = grid.fill_depressions(dem=dem_raw_r)

dem_filled_r = _get_any_layer(grid, preferred_name="dem_filled", before_keys=before)
if dem_filled_r is None:
    if res_fill is None:
        raise RuntimeError("fill_depressions não retornou nada e não registrou nenhuma camada no grid.")
    # Se retornou Raster instance, usar direto; se retornou ndarray, registrar manualmente
    dem_filled_r = res_fill if not isinstance(res_fill, np.ndarray) else _add_layer_from_array(grid, "dem_filled", res_fill, dem_raw_r)

before = _grid_keys(grid)
try:
    res_flats = grid.resolve_flats(dem=dem_filled_r, out_name="dem_hydro")
except TypeError:
    res_flats = grid.resolve_flats(dem=dem_filled_r)

dem_hydro_r = _get_any_layer(grid, preferred_name="dem_hydro", before_keys=before)
if dem_hydro_r is None:
    if res_flats is None:
        raise RuntimeError("resolve_flats não retornou nada e não registrou nenhuma camada no grid.")
    dem_hydro_r = res_flats if not isinstance(res_flats, np.ndarray) else _add_layer_from_array(grid, "dem_hydro", res_flats, dem_raw_r)

dem_hydro = grid.view(dem_hydro_r)

print("DEM hidrologicamente consistente pronto.")





### Verificação visual pós-correção
O DEM hidrológico não deve criar degraus artificiais grandes, mas deve remover armadilhas que bloqueiam o fluxo.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(dem_hydro, cmap="terrain")
plt.colorbar(label="Elevação (m)")
plt.title("DEM após correções hidrológicas (fill + flats)")
plt.tight_layout()
plt.show()


## 5) Direção de fluxo e acumulação
Com o DEM hidrológico, calculamos:
- **Direção de fluxo (D8)**: para onde cada célula drena.
- **Acumulação de fluxo**: quantas células contribuem para cada ponto.

A acumulação é a base para extrair a drenagem: definimos um limiar e consideramos como canal as células com acumulação acima do limiar.

In [ ]:
# Direção de fluxo (D8)
grid.flowdir(data="dem_hydro", out_name="fdir", dirmap=grid.dirmap)

# Acumulação de fluxo
grid.accumulation(data="fdir", out_name="acc")

fdir = grid.view("fdir")
acc  = grid.view("acc")

print("Fluxo e acumulação calculados.")


### Visualização da acumulação
Normalmente usamos escala logarítmica para enxergar a rede de drenagem.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(np.log1p(acc), cmap="viridis")
plt.colorbar(label="log(1 + acumulação)")
plt.title("Acumulação de fluxo (escala log)")
plt.tight_layout()
plt.show()


## 6) Extração de drenagem (canais) via limiar de acumulação
Um canal é definido por `acc >= acc_threshold`.
Esse limiar depende da resolução do DEM e do tamanho mínimo do curso d’água que você quer representar.

Dica prática: varie o limiar e observe se a drenagem fica *densa demais* (limiar baixo) ou *esparsa demais* (limiar alto).

In [ ]:
streams = acc >= acc_threshold

plt.figure(figsize=(8, 6))
plt.imshow(streams, cmap="gray")
plt.title(f"Rede de drenagem (acc >= {acc_threshold})")
plt.tight_layout()
plt.show()

print("Proporção de células classificadas como canal:", float(np.mean(streams)))


## 7) Cálculo do HAND (Height Above Nearest Drainage)
O HAND é a diferença de elevação entre uma célula e a célula do canal mais próxima **ao longo do caminho de escoamento**.
Em termos simples:
- para cada ponto, seguimos o fluxo até encontrar um canal;
- o HAND é `elevação(ponto) - elevação(canal)`.

O `pysheds` oferece uma função `hand` em versões recentes. Caso a sua versão não tenha, a célula abaixo inclui uma alternativa robusta:
1) *snap* de cada célula ao canal ao longo do fluxo,
2) obter elevação do canal de referência,
3) subtrair.

> Se o seu script original já calcula HAND por outro método (por exemplo, TauDEM/Whitebox), substitua esta etapa mantendo a lógica e as saídas.

In [ ]:
# Tentativa 1: usar diretamente grid.hand (se disponível)
hand = None
if hasattr(grid, "hand"):
    try:
        grid.hand(data="dem_hydro", flowdir="fdir", streams=streams, out_name="hand")
        hand = grid.view("hand")
        print("HAND calculado com grid.hand().")
    except Exception as e:
        print("Falha ao usar grid.hand(). Vamos tentar método alternativo.")
        print("Erro:", e)

# Método alternativo (fallback)
if hand is None:
    # Identificar células de canal como "alvos" para snap ao longo do fluxo
    # pysheds tem função "snap_to_mask" (em algumas versões) ou "catchment"/"distance_to_outlet".
    # Aqui implementamos uma estratégia segura:
    # - Criar um raster 'channel_elev' contendo elevação do DEM apenas nos canais (NaN no resto)
    dem_arr = dem_hydro.astype("float32")
    channel_elev = np.where(streams, dem_arr, np.nan).astype("float32")

    # Para cada célula, queremos a elevação do canal atingido seguindo fdir.
    # Implementação iterativa com compressão de caminho (path compression) para ser viável.
    # Observação: em rasters grandes pode ser pesado; prefira grid.hand() ou ferramentas otimizadas.
    fdir_arr = fdir.copy()
    nrows, ncols = dem_arr.shape

    # Mapeamento de direção D8 para deslocamento (dr, dc) conforme dirmap do pysheds
    # grid.dirmap define o valor -> (dr, dc)
    # Vamos construir um dicionário valor->(dr, dc)
    dirmap = dict(zip(grid.dirmap, [(-1, 0), (-1, 1), (0, 1), (1, 1),
                                   (1, 0), (1, -1), (0, -1), (-1, -1)]))

    # Funções auxiliares para indexação linear
    def to_lin(r, c): return r * ncols + c
    def to_rc(idx): return divmod(idx, ncols)

    # Inicializar "representante do canal" para cada célula (índice linear do canal atingido)
    rep = np.full(nrows * ncols, -1, dtype=np.int64)

    # Marcar canais: representante é ele mesmo
    stream_idx = np.flatnonzero(streams.ravel())
    rep[stream_idx] = stream_idx

    # Máscara de células válidas (não-NaN)
    valid = ~np.isnan(dem_arr.ravel())

    # Função para seguir fluxo até canal (com memoização)
    sys.setrecursionlimit(10_000_000)

    def find_rep(i):
        # Se já sabemos ou é inválido, retorna
        if not valid[i]:
            return -1
        if rep[i] != -1:
            return rep[i]
        r, c = to_rc(i)
        dval = fdir_arr[r, c]
        if dval not in dirmap:
            rep[i] = -1
            return -1
        dr, dc = dirmap[dval]
        nr, nc = r + dr, c + dc
        if nr < 0 or nr >= nrows or nc < 0 or nc >= ncols:
            rep[i] = -1
            return -1
        j = to_lin(nr, nc)
        root = find_rep(j)
        rep[i] = root  # path compression
        return root

    # Executar para todos os pixels válidos
    for i in range(nrows * ncols):
        if rep[i] == -1 and valid[i]:
            find_rep(i)

    # Elevação do canal atingido
    rep_rc = np.array([to_rc(x) if x != -1 else (-1, -1) for x in rep], dtype=np.int32)
    chan_elev = np.full(nrows * ncols, np.nan, dtype=np.float32)
    ok = rep != -1
    chan_elev[ok] = dem_arr[rep_rc[ok,0], rep_rc[ok,1]]

    # HAND = elevação - elevação do canal
    hand = (dem_arr.ravel() - chan_elev).reshape((nrows, ncols)).astype(np.float32)
    hand = np.where(valid.reshape((nrows,ncols)), hand, np.nan)
    print("HAND calculado com método alternativo (fallback).")


NameError: name 'grid' is not defined

### Visualização do HAND
Valores baixos (próximos de 0 m) tendem a indicar proximidade altimétrica com a drenagem e, em geral, maior suscetibilidade à inundação (dependendo do contexto hidrológico).

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(hand, cmap="magma", vmin=0, vmax=np.nanpercentile(hand, 99))
plt.colorbar(label="HAND (m)")
plt.title("HAND (altura acima da drenagem mais próxima)")
plt.tight_layout()
plt.show()

print("Estatísticas (m):",
      "min=", float(np.nanmin(hand)),
      "p50=", float(np.nanpercentile(hand, 50)),
      "p90=", float(np.nanpercentile(hand, 90)),
      "max=", float(np.nanmax(hand)))


## 8) Classificação de risco/suscetibilidade a partir do HAND (opcional)
Um uso comum do HAND é gerar classes discretas (por exemplo, 0–2 m = muito alto, 2–5 m = alto, etc.).
Essas classes são **heurísticas** e devem ser ajustadas ao contexto local, histórico de cheias e validação.

Abaixo criamos um raster categórico de classes com base em `hand_classes_m`.

In [ ]:
# Converter limites em intervalos e atribuir classes inteiras
bins = np.array(hand_classes_m, dtype=float)
# classes: 1..(len(bins)-1)
hand_class = np.digitize(hand, bins=bins, right=False).astype(np.uint8)

# Tornar NaN como 0 (classe 0 = sem dado)
hand_class = np.where(np.isfinite(hand), hand_class, 0).astype(np.uint8)

plt.figure(figsize=(8, 6))
plt.imshow(hand_class, cmap="tab20")
plt.colorbar(label="Classe HAND (inteira)")
plt.title("Classes de suscetibilidade (baseadas em HAND)")
plt.tight_layout()
plt.show()

unique, counts = np.unique(hand_class, return_counts=True)
print("Distribuição de classes (classe: contagem):")
for u, c in zip(unique, counts):
    print(int(u), int(c))


## 9) Exportação dos rasters (HAND e classes)
Nesta etapa salvamos as saídas em GeoTIFF.
É importante definir corretamente `dtype`, `nodata` e manter o `transform` e CRS do DEM original.

In [ ]:
def write_geotiff(out_fp: Path, arr: np.ndarray, ref_profile: dict, dtype, nodata):
    prof = ref_profile.copy()
    prof.update(count=1, dtype=dtype, nodata=nodata, compress="deflate")
    with rasterio.open(out_fp, "w", **prof) as dst:
        dst.write(arr, 1)

# HAND (float32) com nodata = -9999
hand_out = np.where(np.isfinite(hand), hand, -9999).astype(np.float32)
write_geotiff(OUT_DIR / "hand.tif", hand_out, dem_profile, dtype="float32", nodata=-9999)

# Classes (uint8) com nodata = 0
write_geotiff(OUT_DIR / "hand_classes.tif", hand_class.astype(np.uint8), dem_profile, dtype="uint8", nodata=0)

print("Arquivos gerados:")
print(" -", OUT_DIR / "hand.tif")
print(" -", OUT_DIR / "hand_classes.tif")


## 10) Próximos passos (dependendo do seu script)
Conforme o seu pipeline HAND, é comum adicionar:
- Validação com manchas históricas de inundação (se existir).
- Integração com chuva, nível do rio, ou um modelo hidrodinâmico simplificado.
- Calibração do limiar de canais e das classes HAND.

Se você colar o seu script, eu adapto este notebook para reproduzir **exatamente**:
- as mesmas bibliotecas,
- os mesmos parâmetros,
- o mesmo formato de saída,
- e a mesma lógica do seu método HAND (inclusive particularidades de pré-processamento).